# Chapter 5 — Defending with Agents: Autonomous SecOps

## Objective

Assemble an autonomous, auditable SecOps pipeline — Planner-Executor-Verifier with a human gate, an event-bus audit trail, and signed events.

By the end you will be able to:

- Build a Planner → Executor → Verifier triage pipeline with a human gate
- Explain why an append-only log is a forensic audit trail
- Turn free-text alerts into structured tickets
- Emit signed, hash-chained audit events
- Apply the reversible-auto / irreversible-human-gate rule

## Summary

This notebook turns the previous attacks into defenses. You assemble an autonomous, auditable security-operations pipeline: a Planner assesses an alert, an Executor enriches and acts on it, and a Verifier validates the outcome and refuses to let an irreversible action through without a human. Around that pipeline you add an ordered event bus that doubles as a tamper-evident audit trail, a language-model helper that turns free-text alerts into structured tickets, and signed, hash-chained audit events. The guiding rule is bounded autonomy: reversible actions run at machine speed, irreversible ones stop for a person.

## Outline

**Chapter arc:** SecOps foundations → AI-based security processes → agentic SecOps workflows → AI SIEM, SOAR & security copilots

This notebook covers:

- Anomaly detection with a reconstruction scorer
- The Planner - Executor - Verifier pipeline
- The event backbone and audit trail
- The incident-reporting chatbot
- Runnable demos plus a live language-model example
- A self-check quiz

## The big picture

Defense turns the previous attacks into an **autonomous, auditable SecOps pipeline**. Every SOC runs the same spine — collection → detection → triage → investigation → response — and AI augments each stage.

- **Planner → Executor → Verifier:** the Planner assesses severity, the Executor enriches and acts, the Verifier validates and gates.
- **Bounded autonomy:** reversible actions (block an IP, isolate an endpoint) run at machine speed; irreversible ones (wipe, delete, deprovision) stop for a human.
- **The audit trail:** an ordered, append-only event log is both the inter-agent bus and tamper-evident evidence; every action emits a signed event.
- **Measure it:** drive mean-time-to-detect and mean-time-to-respond down, and let every incident tune the next — the self-improving '10X SOC'.

**Planner - Executor - Verifier**

<svg viewBox="0 0 592 142" width="100%" style="max-width:592px;height:auto;font-family:system-ui,-apple-system,Segoe UI,Arial,sans-serif" xmlns="http://www.w3.org/2000/svg" role="img"><defs><marker id="a" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#64748b"/></marker></defs><rect width="592" height="142" rx="12" fill="#ffffff"/><rect x="22" y="22" width="156" height="62" rx="9" fill="#1d4ed8"/><text x="100" y="45" fill="#fff" font-size="15" font-weight="600" text-anchor="middle" dominant-baseline="middle">Planner</text><text x="100" y="63" fill="#e2e8f0" font-size="10.5" text-anchor="middle" dominant-baseline="middle">assess severity</text><line x1="178" y1="53" x2="218" y2="53" stroke="#64748b" stroke-width="2" marker-end="url(#a)"/><rect x="218" y="22" width="156" height="62" rx="9" fill="#0f766e"/><text x="296" y="45" fill="#fff" font-size="15" font-weight="600" text-anchor="middle" dominant-baseline="middle">Executor</text><text x="296" y="63" fill="#e2e8f0" font-size="10.5" text-anchor="middle" dominant-baseline="middle">enrich + act</text><line x1="374" y1="53" x2="414" y2="53" stroke="#64748b" stroke-width="2" marker-end="url(#a)"/><rect x="414" y="22" width="156" height="62" rx="9" fill="#b91c1c"/><text x="492" y="45" fill="#fff" font-size="15" font-weight="600" text-anchor="middle" dominant-baseline="middle">Verifier</text><text x="492" y="63" fill="#e2e8f0" font-size="10.5" text-anchor="middle" dominant-baseline="middle">validate / gate</text><text x="296" y="106" fill="#334155" font-size="12.5" text-anchor="middle">Reversible actions auto-run; irreversible ones stop at a human gate. Every step</text><text x="296" y="124" fill="#334155" font-size="12.5" text-anchor="middle">is a signed, append-only audit event.</text></svg>

## Setup

This notebook runs on **numpy** and **scikit-learn**. Some functions integrate a message bus (**kafka**), a web API (**fastapi**), or forensic tooling, imported lazily inside them. Every core example has an offline path, so no key is required; one is wired below for the live extractor demo.

In [1]:
import os

# Load the OpenAI key into the environment (never printed): the key file on
# the lab VM, or OPENAI_API_KEY already set in the environment.
KEY_PATH = '/home/student/keys/key.txt'
client = None
if os.path.exists(KEY_PATH):
    os.environ["OPENAI_API_KEY"] = open(KEY_PATH).read().strip()
if os.environ.get("OPENAI_API_KEY"):
    try:
        from openai import OpenAI
        client = OpenAI()
        print("OpenAI key loaded — client ready (key not shown).")
    except Exception as e:
        print("openai SDK unavailable; the live demo will be skipped:", e)
else:
    print(f"No key file at {KEY_PATH} and no OPENAI_API_KEY — the live demo will be skipped.")


OpenAI key loaded — client ready (key not shown).


Imports used across the examples:

In [2]:
import numpy as np

## Core building blocks

Each function below is introduced, then defined. Run them all, then see them in action in **Try it** and the **Live demo**.

### Anomaly detection with a reconstruction scorer

The detector (`ReconAnomalyScorer`) raises an alert; the **Planner** assesses severity, the **Executor** enriches it with VirusTotal + CVE and acts, the **Verifier** validates and rolls back irreversible actions that lack human approval.

> **Security lens:** train on a benign baseline; a high reconstruction error means 'far from normal' — the alert the Planner triages.

In [3]:
class ReconAnomalyScorer:
    """Stand-in for a trained VAE using PCA encode/decode for portability."""
    def __init__(self, n_components=6, pct=99):
        from sklearn.decomposition import PCA
        self.pca = PCA(n_components=n_components)
        self.pct = pct
        self.threshold = None

    def fit(self, benign):
        self.pca.fit(benign)
        self.threshold = np.percentile(self._err(benign), self.pct)
        return self

    def _err(self, X):
        Xr = self.pca.inverse_transform(self.pca.transform(X))
        return ((X - Xr) ** 2).mean(axis=1)

    def score(self, x):
        e = float(self._err(np.atleast_2d(x))[0])
        return {"score": e, "anomaly": bool(e > self.threshold)}


### The Planner - Executor - Verifier pipeline

The triage pipeline: the Planner assesses severity, the Executor enriches and acts, and the Verifier validates the outcome — rolling back any irreversible action a human has not approved.

> **Security lens:** the Verifier is the control point — policy checks, output validation, and the human gate on irreversible actions all live there.

In [4]:
def planner(alert):
    sev = "high" if alert["score"] >= 0.8 else "low"
    return {"severity": sev,
            "action": "quarantine" if sev == "high" else "monitor"}

def executor(plan, run_tool=None):
    # The Executor enriches via its tool belt (VirusTotal / a CVE lookup /
    # firewall tools) before acting, and flags whether the action is
    # irreversible (so the Verifier can gate it).
    enrichment = run_tool(plan) if run_tool else None
    irreversible = plan["action"] in {"wipe", "delete"}
    return {"action": plan["action"], "done": True,
            "irreversible": irreversible, "enrichment": enrichment}

def verifier(result, human_ok=lambda r: True):
    # Human-on-the-loop gate: an irreversible action with no approval rolls back.
    if result["irreversible"] and not human_ok(result):
        return {**result, "ok": False, "rolled_back": True}
    return {**result, "ok": result["done"], "rolled_back": False}

def pev_pipeline(alert, human_ok=lambda r: True, run_tool=None):
    plan = planner(alert)
    return verifier(executor(plan, run_tool), human_ok)


### The event backbone and audit trail

A durable, **ordered** event bus between the detector agent (producer) and the triage agent (consumer). The append-only topic log is the forensic audit trail.

> **Security lens:** an ordered, append-only topic log doubles as a tamper-evident, replayable audit trail — the same log governance later audits.

In [5]:
def publish_alert(alert, topic="alerts", servers="localhost:9092"):
    from kafka import KafkaProducer
    import json
    prod = KafkaProducer(bootstrap_servers=servers,
                         value_serializer=lambda v: json.dumps(v).encode())
    prod.send(topic, alert)        # every message is an ordered audit record
    prod.flush()

def consume_alerts(handler, topic="alerts", group="triage", servers="localhost:9092"):
    from kafka import KafkaConsumer
    import json
    for msg in KafkaConsumer(topic, group_id=group, bootstrap_servers=servers):
        handler(json.loads(msg.value))
class LocalBus:
    """In-process stand-in for the broker — same publish/consume shape over an
    append-only log, so the audit-trail flow runs with no service running.
    Swap in publish_alert/consume_alerts (Kafka) in production."""
    def __init__(self): self.topics = {}
    def publish(self, topic, event):
        self.topics.setdefault(topic, []).append(event)   # append-only, ordered
    def consume(self, topic, handler):
        for event in self.topics.get(topic, []):
            handler(event)


### The incident-reporting chatbot

Turn a free-text incident report into a structured JSON ticket using the lab's `extract_entities`, exposed behind a FastAPI `/report` endpoint.

> **Security lens:** free text in, structured ticket out — the contract that feeds the Planner and the event bus stays the same whether a regex or a model extracts.

In [6]:
import re
def extract_entities(text):
    host = re.search(r"\b(host[-\s]?\w+|\d+\.\d+\.\d+\.\d+)\b", text, re.I)
    sev  = re.search(r"\b(critical|high|medium|low)\b", text, re.I)
    return {
        "host":     host.group(0) if host else None,
        "severity": sev.group(0).title() if sev else None,
        "ioc":      re.findall(r"\b[a-f0-9]{32,64}\b", text, re.I),
    }

## Try it

Run the self-contained demos and read the output: the reconstruction scorer flags a synthetic attack point, its **real score** drives the Planner-Executor-Verifier pipeline, the audit trail replays over the local bus, and the regex extractor parses an alert.


In [7]:
# The detector's real output drives the rest of the notebook.
rng = np.random.default_rng(0)
scorer = ReconAnomalyScorer().fit(rng.normal(0, 1, size=(500, 16)))
benign_pt = rng.normal(0, 1, size=16)
attack_pt = rng.normal(3, 2, size=16)
print("anomaly (benign):", scorer.score(benign_pt))
print("anomaly (attack):", scorer.score(attack_pt))

# Calibrate: score = reconstruction error as a fraction of twice the
# anomaly threshold, capped at 0.99 — above-threshold traffic triages high.
def to_alert(host, pt):
    e = scorer.score(pt)["score"]
    return {"host": host, "score": round(float(min(0.99, e / (2 * scorer.threshold))), 2)}

det_alert  = to_alert("host-DB7", attack_pt)
det_alert2 = to_alert("host-W12", benign_pt)
print("\nalert from detector:", det_alert)
print("PEV (attack):", pev_pipeline(det_alert))
print("PEV (benign):", pev_pipeline(det_alert2))
print("PEV (irreversible, denied):",
      verifier({"action": "wipe", "done": True, "irreversible": True,
                "enrichment": None}, human_ok=lambda r: False))

# The audit trail over the local bus: publish, then replay in order.
bus = LocalBus()
for a in (det_alert, det_alert2):
    bus.publish("alerts", a)
log = []
bus.consume("alerts", log.append)
print("\nbus audit log:", log)

print("\nextract:", extract_entities(
    "Critical alert on host-DB7, hash deadbeefcafe1234deadbeefcafe1234"))


anomaly (benign): {'score': 0.3512349889913815, 'anomaly': False}
anomaly (attack): {'score': 5.393077802146983, 'anomaly': True}

alert from detector: {'host': 'host-DB7', 'score': 0.99}
PEV (attack): {'action': 'quarantine', 'done': True, 'irreversible': False, 'enrichment': None, 'ok': True, 'rolled_back': False}
PEV (benign): {'action': 'monitor', 'done': True, 'irreversible': False, 'enrichment': None, 'ok': True, 'rolled_back': False}
PEV (irreversible, denied): {'action': 'wipe', 'done': True, 'irreversible': True, 'enrichment': None, 'ok': False, 'rolled_back': True}

bus audit log: [{'host': 'host-DB7', 'score': 0.99}, {'host': 'host-W12', 'score': 0.15}]

extract: {'host': 'host-DB7', 'severity': 'Critical', 'ioc': ['deadbeefcafe1234deadbeefcafe1234']}


## Live demo — the LLM incident-report extractor

`extract_entities` uses a regex stand-in so it runs with no model. With the key wired, here is the language-model version that parses messier free-text alerts into the same structured ticket. Requires the API key wired above.

In [8]:
import json

def extract_entities_llm(text, client, model="gpt-4o-mini"):
    prompt = ("Extract fields from this security alert and return JSON with keys "
              "host, severity (Critical/High/Medium/Low), and ioc (list of hex hash "
              "strings). Alert:\n" + text)
    r = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}, temperature=0, max_tokens=200)
    return json.loads(r.choices[0].message.content)

alert = "Critical alert on host-DB7, hash deadbeefcafe1234deadbeefcafe1234"
print("regex stand-in :", extract_entities(alert))
if client:
    print("LLM extractor  :", extract_entities_llm(alert, client))
else:
    print("LLM extractor  : skipped (no API key)")

regex stand-in : {'host': 'host-DB7', 'severity': 'Critical', 'ioc': ['deadbeefcafe1234deadbeefcafe1234']}


LLM extractor  : {'host': 'DB7', 'severity': 'Critical', 'ioc': ['deadbeefcafe1234deadbeefcafe1234']}


### Executor enrichment and a GenAI SOAR playbook

Before acting, the Executor enriches an alert with VirusTotal + CVE context (mocked so it runs offline), and a GenAI step proposes a Volatility playbook — with a fixed, reviewable fallback when no model is wired.

In [9]:
def vt_lookup(ioc):   return {"ioc": ioc, "malicious": 3, "vendors": 70,
                              "source": "VirusTotal (mock — no API key offline)"}

CVE_MINI_DB = {          # bundled, real CVEs so the lookup works offline
    "log4j":   {"cve": "CVE-2021-44228", "cvss": 10.0, "name": "Log4Shell"},
    "xz":      {"cve": "CVE-2024-3094",  "cvss": 10.0, "name": "xz-utils backdoor"},
    "openssl": {"cve": "CVE-2014-0160",  "cvss": 7.5,  "name": "Heartbleed"},
}
def cve_lookup(product):
    hit = CVE_MINI_DB.get(product.lower())
    return {"product": product,
            **(hit or {"cve": None, "note": "not in the bundled mini-DB"})}

def soc_tools(plan):     # the Executor's tool belt
    return {"vt":  vt_lookup("deadbeefcafe1234deadbeefcafe1234"),
            "cve": cve_lookup("log4j")}

# det_alert from the Try-it detector run above; the Executor now enriches for real.
print("decision:", pev_pipeline(det_alert, run_tool=soc_tools))
print("cve miss :", cve_lookup("not-in-db"))

import json
def gen_playbook(llm=None):
    if llm is None:                            # offline fallback
        return [{"plugin": "windows.pslist",  "why": "enumerate processes"},
                {"plugin": "windows.malfind", "why": "find injected code"},
                {"plugin": "windows.netscan", "why": "surface C2 connections"}]
    return json.loads(llm("Return a JSON list of Volatility 3 plugins to triage a Windows memory "
                          "image; each item has 'plugin' and 'why'."))
print("playbook:", json.dumps(gen_playbook(), indent=2))


decision: {'action': 'quarantine', 'done': True, 'irreversible': False, 'enrichment': {'vt': {'ioc': 'deadbeefcafe1234deadbeefcafe1234', 'malicious': 3, 'vendors': 70, 'source': 'VirusTotal (mock — no API key offline)'}, 'cve': {'product': 'log4j', 'cve': 'CVE-2021-44228', 'cvss': 10.0, 'name': 'Log4Shell'}}, 'ok': True, 'rolled_back': False}
cve miss : {'product': 'not-in-db', 'cve': None, 'note': 'not in the bundled mini-DB'}
playbook: [
  {
    "plugin": "windows.pslist",
    "why": "enumerate processes"
  },
  {
    "plugin": "windows.malfind",
    "why": "find injected code"
  },
  {
    "plugin": "windows.netscan",
    "why": "surface C2 connections"
  }
]


### Zero-Trust signed audit events

Every agent action emits a signed, hash-chained event before the next step runs: each event's `prev` field is the previous event's signature, so editing any event breaks every signature after it. Reversible actions auto-run; the irreversible `wipe` is rolled back with no human approval.

In [10]:
import hmac, hashlib, json

_AUDIT_KEY = b"lab-audit-key"        # in production: a per-agent key in a secrets manager / HSM
_chain = {"prev": "0" * 64}          # genesis: the first event chains to all-zeros

def sign_event(agent, action, outcome, ts="2026-01-01T00:00:00Z"):
    ev = {"agent": agent, "action": action, "outcome": outcome, "ts": ts, "prev": _chain["prev"]}
    ev["sig"] = hmac.new(_AUDIT_KEY, json.dumps(ev, sort_keys=True).encode(), hashlib.sha256).hexdigest()
    _chain["prev"] = ev["sig"]
    return ev

def run_signed(plan, human_ok=lambda r: True):
    result = executor(plan)
    evs = [sign_event("executor", plan["action"], "done" if result["done"] else "fail")]
    verdict = verifier(result, human_ok)
    evs.append(sign_event("verifier", "verify", "ok" if verdict["ok"] else "rolled_back"))
    return evs

plan = planner(det_alert)   # the detector alert from Try it
events = [sign_event("planner", "plan", plan["action"])]
events += run_signed(plan)                                        # reversible -> auto-runs
events += run_signed({"action": "wipe"}, human_ok=lambda r: False)  # irreversible -> rolled back
for ev in events:
    print(f"{ev['agent']:9} {ev['action']:11} {ev['outcome']:11} prev={ev['prev'][:8]}.. sig={ev['sig'][:8]}..")
print("chain intact:", all(events[i]['sig'] == events[i+1]['prev'] for i in range(len(events) - 1)))


planner   plan        quarantine  prev=00000000.. sig=e741e260..
executor  quarantine  done        prev=e741e260.. sig=f5921f7f..
verifier  verify      ok          prev=f5921f7f.. sig=0ca67315..
executor  wipe        done        prev=0ca67315.. sig=10dbebd1..
verifier  verify      rolled_back prev=10dbebd1.. sig=c2c42213..
chain intact: True


## Quiz

Answer each question, then fill in the `answers` dict in the next cell (`"A"`, `"B"`, …) and run it to grade yourself.

**1. The Verifier is a human-on-the-loop gate. Which actions does it hold for approval?**

- A. All actions
- B. Reversible actions
- C. Irreversible actions with no human approval — it rolls them back
- D. Read-only actions

**2. Why does an append-only Kafka topic log double as a forensic audit trail?**

- A. It is encrypted
- B. Its records are ordered and immutable — replaying from the start yields the identical sequence
- C. It is fast
- D. It compresses well

**3. Under reversible-auto / irreversible-human-gate, which runs at machine speed?**

- A. Irreversible actions
- B. Reversible containment (e.g., isolate an endpoint)
- C. Nothing
- D. Only human-approved actions

**4. The signed audit events form a hash chain. How?**

- A. Each event is random
- B. Each event's prev equals the previous event's signature — editing one breaks all after it
- C. They share one key only
- D. They are timestamped only

**5. ReconAnomalyScorer is trained on what, and what signals an anomaly?**

- A. Attack traffic; low error
- B. A benign baseline; a high reconstruction error means 'far from normal'
- C. Random data; any output
- D. Labeled data; a probability


In [11]:
import base64
_KEY = {'q1': 'Qw==', 'q2': 'Qg==', 'q3': 'Qg==', 'q4': 'Qg==', 'q5': 'Qg=='}   # base64 so the answers aren't obvious at a glance
_WHY = {'q1': 'Irreversible, unapproved actions are rolled back.', 'q2': 'Ordered, append-only history is tamper-evident and replayable.', 'q3': 'Reversible containment auto-runs; irreversible actions wait for a human.', 'q4': 'Chaining prev→sig makes any edit detectable.', 'q5': 'Train on normal, flag high reconstruction error.'}

def check(answers):
    key = {q: base64.b64decode(v).decode() for q, v in _KEY.items()}
    if all(str(answers.get(q, '')).strip().upper() in ('', '?') for q in key):
        print('Fill in the answers dict above (A/B/C/D) and re-run to grade.')
        return
    score = 0
    for q in sorted(key):
        got = str(answers.get(q, '')).strip().upper()
        if got == key[q]:
            score += 1; print(f'\u2713 {q}: {got}')
        else:
            print(f'\u2717 {q}: you put {got or "\u2014"} \u2014 correct is {key[q]}. {_WHY[q]}')
    print(f'\nScore: {score}/{len(key)}')

In [12]:
answers = {
    "q1": "?",   # A / B / C / D
    "q2": "?",   # A / B / C / D
    "q3": "?",   # A / B / C / D
    "q4": "?",   # A / B / C / D
    "q5": "?",   # A / B / C / D
}
check(answers)

Fill in the answers dict above (A/B/C/D) and re-run to grade.


## Key takeaways

- Defense is a Planner → Executor → Verifier pipeline: reversible actions auto-run, irreversible ones stop for a human.
- An ordered, append-only event log is both the inter-agent bus and the tamper-evident audit trail.
- Signed, hash-chained events make the trail evidence: altering one breaks every signature after it.
- Detection compresses time-to-detect and time-to-respond; every outcome feeds back to tune the detectors.